<a href="https://colab.research.google.com/github/cuiandrew08-lab/LiDARFusionLearning/blob/main/CenterPointHead.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import numpy as np

from google.colab import drive
#drive.mount("/content/drive")

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms as transforms

import tensorflow as tf

TORCH_version = torch.__version__.split('+')[0]
CUDA_version = torch.version.cuda.replace('.', '')

!pip install torch-scatter torch-sparse torch-cluster -f https://data.pyg.org/whl/torch-{TORCH_version}+cu{CUDA_version}.html

import torch_sparse

from scipy.ndimage import maximum_filter

Looking in links: https://data.pyg.org/whl/torch-2.11.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 97.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 69.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 79.4 MB/s eta 0:00:00


In [ ]:
def generate_heatmap(w,h, centers,sigma):

  x_axis = torch.arange(0, w, device=centers.device, dtype=torch.float32)
  y_axis = torch.arange(0, h, device=centers.device, dtype=torch.float32)
  y_grid, x_grid = torch.meshgrid(y_axis, x_axis, indexing='ij')

  distances_squared = torch.zeros((w,h), device=centers.device)

  for i in range(centers.shape[0]):
    distances_squared += (x_grid-centers[i][0])**2 +(y_grid-centers[i][1])

  heatmap = torch.exp(-1*distances_squared/(2*sigma**2))

  return heatmap

In [41]:
class CenterPoint_FirstStage(nn.Module): #similar classes(num_classes) are grouped together into one task head. The classes should have similar size, z, yaw, etc. so one task head can be used
#input is assumed to be of shape [C,W,H]
  def __init__(self, w, h, C, num_classes, K, hidden_channels=64):
    super().__init__()

    self.w = w
    self.h = h
    self.num_classes = num_classes
    self.C = C
    self.K = K #K is max number of centers per heatmap layer

    self.shared = nn.Sequential(nn.Conv2d(self.C, hidden_channels, kernel_size = 3, padding = 1, bias = True), nn.ReLU())

    self.heat_conv = nn.Conv2d(hidden_channels, num_classes, kernel_size = 1)

    self.offset = nn.Conv2d(hidden_channels, 2, kernel_size = 1)             # (δx, δy)
    self.z = nn.Conv2d(hidden_channels, 1, kernel_size = 1)             # z center
    self.size = nn.Conv2d(hidden_channels, 3, kernel_size = 1)             # log(w, l, h)
    self.yaw = nn.Conv2d(hidden_channels, 2, kernel_size = 1)             # (sin θ, cos θ)
    #self.velocity = nn.Conv2d(hidden_channels, 2, kernel_size = 1)


  def get_shared_feats(self, feature_map):
    return self.shared(feature_map)

  def learn_heatmap(self, feature_map):

    heatmap = self.shared(feature_map)
    heatmap = F.sigmoid(self.heat_conv(heatmap)).permute(1,2,0)

    return heatmap

  def get_centers(self, heatmap):
    centers_list = []

    for k in range(self.num_classes):
      heatmap_k = heatmap[:,:, k].cpu().detach().numpy()

      local_max_filter = maximum_filter(heatmap_k, size=3, mode='constant', cval=-np.inf)

      is_local_max = (heatmap_k == local_max_filter)

      rows, cols = np.where(is_local_max)

      for i in range(rows.size):

        layer_center_list = []
        I_list = []

        I = heatmap_k[rows[i]][cols[i]]

        center_0 = torch.tensor([rows[i],cols[i],I,k])

        I_list.append(I)
        layer_center_list.append(center_0)

      while len(layer_center_list) > self.K:
        j = I_list.index(min(I_list))
        layer_center_list.remove(layer_center_list[j])

      layer_center_list = torch.stack(layer_center_list)

      centers_list.append(layer_center_list)

    centers = torch.cat(centers_list, dim = 0)

    return centers

  def regression_heads(self, feature_map):

    shared_feats = self.shared(feature_map)

    offset = self.offset(shared_feats).permute(1,2,0)
    z = self.z(shared_feats).squeeze()
    size = self.size(shared_feats).permute(1,2,0)
    yaw = self.yaw(shared_feats).permute(1,2,0)
    #velocity = self.velocity(shared_feats).permute(1,2,0)

    return offset, z, size, yaw

  def forward(self, feature_map):

    heatmap = self.learn_heatmap(feature_map)

    offset, z, size, yaw = self.regression_heads(feature_map)

    centers = self.get_centers(heatmap)

    out_list = []

    for i in range(centers.shape[0]):

      cx = int(centers[i][0])
      cy = int(centers[i][1])
      I = centers[i][2]
      k = centers[i][3]

      offset_0 = offset[cx][cy]
      z_0 = z[cx][cy].detach()
      size_0 = size[cx][cy]
      yaw_0 = yaw[cx][cy]

      temp_tensor = torch.tensor([cx, cy, z_0, k, I], dtype = offset_0.dtype, device = offset_0.device)

      out_0 = [size_0, yaw_0, temp_tensor, offset_0] #test first

      out_0 = torch.cat(out_0)

      out_list.append(out_0)

    out = torch.stack(out_list, dim = 0)

    return out



In [50]:
class CenterPoint_SecondStage(nn.Module):

  def __init__(self, w, h, C, num_classes, hidden_channels = 32):

    super().__init__()

    self.w = w
    self.h = h
    self.C =C
    self.num_classes = num_classes

    self.refine_network = nn.Sequential(nn.Linear(5*C, hidden_channels, bias = True),
                                        nn.ReLU(),
                                        nn.Linear(hidden_channels,8))

  def get_box_centers(self, size, yaw, center):

    lwh = torch.exp(size)
    l_1 = (yaw * lwh[0]*0.5) + center
    l_2 = l_1 * -1
    l_3 = (yaw * lwh[1]*0.5)+ center
    l_4 = l_3 * -1

    center_list = [center,l_1,l_2,l_3,l_4]

    return torch.stack(center_list)

  def BEV_sample(self, box_centers, shared_map): #box centers should be batched - [N, 5, 2]

    C = shared_map.shape[0]
    N = box_centers.shape[0]

    max_x = torch.max(box_centers[:,:,0])
    max_y = torch.max(box_centers[:,:,1])

    min_x = torch.min(box_centers[:,:,0])
    min_y = torch.min(box_centers[:,:,1])

    normalized_x = 2*((box_centers[:,:,0]+min_x)/(max_x+min_x)) -1
    normalized_y = 2*((box_centers[:,:,1]+min_y)/(max_y+min_y)) -1

    normalized_centers = torch.stack([normalized_x, normalized_y]).permute(1,2,0)

    sampled = F.grid_sample(shared_map.unsqueeze(0), normalized_centers.unsqueeze(0), align_corners=True)

    out_feats = sampled.view(N, 5*C)

    return out_feats

  def refine_boxes(self, sampled_centers, original_boxes):

    out_list = []

    out_refine = self.refine_network(sampled_centers)

    dx, dy, dz, dlogl, dlogw, dlogh, dyaw, I = out_refine.T

    for i in range(original_boxes.shape[0]):

      d_vec = torch.tensor([dlogl[i],dlogw[i],dlogh[i],0,0,dx,dy,dz,0,0,0,0])

      alpha = torch.arcsin(original_boxes[i][3])
      alpha = alpha+dyaw[i]
      sin_0 = torch.sin(alpha)
      cos_0 = torch.cos(alpha)

      I_0 = torch.sqrt(I[i]*original_boxes[i][9])

      out_0 = original_boxes[i] + d_vec

      out_0[3] = sin_0.item()
      out_0[4] = cos_0.item()
      out_0[9] = I_0.item()

      out_list.append(out_0)

    out = torch.stack(out_list)

    return out


In [37]:
rng = np.random.default_rng()

test_mat = rng.random((20,10,10))

test_mat = torch.from_numpy(test_mat).to(torch.float)

In [35]:
t_1= torch.ones([3,2])
t_2 =torch.zeros([3,3])

x,y = t_1.T

In [36]:
x

tensor([1., 1., 1.])

In [42]:
device = "cuda"

test_mat = test_mat.to(device)

In [43]:
model = CenterPoint_FirstStage(10,10,20, 4, 100).to(device)

In [44]:
out = model(test_mat)

In [45]:
model.get_shared_feats(test_mat).shape

torch.Size([64, 10, 10])

In [52]:
model_2 = CenterPoint_SecondStage(10,10,20,4).to(device)